In [1]:
import eikon as ek
# ek.set_app_key("66a448a7f40a4da88f6871950ed6eb9dcbee3812")
ek.set_app_key("cd616ce2f9e2497ea982d123ce53db5abf924b9d")
import numpy
import pandas as pd
import refinitiv.data as rd
from refinitiv.data.content import symbol_conversion
numpy.version.version
from threading import Event
timer = Event()

In [2]:
data_df = pd.read_csv("ticker_names_unique.csv")
isin_numbers = data_df['ISIN number'].to_list()
rics_list = []
isin_chunks = [isin_numbers[x:x+500] for x in range(0, len(isin_numbers), 200)]

In [3]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x12806c8f0 {name='workspace'}>

In [4]:
for i in range(3):
    response = symbol_conversion.Definition(
        symbols=isin_chunks[i],
        from_symbol_type=symbol_conversion.SymbolTypes.ISIN,
        to_symbol_types=[
            symbol_conversion.SymbolTypes.RIC,
            symbol_conversion.SymbolTypes.OA_PERM_ID
        ],
    ).get_data()
    rics = response.data.df.RIC.to_list()
    rics_list.append(rics)
    stock_dict = {}
    for ric in rics:
        try:
            stock_dict[ric] = ek.get_timeseries(ric, start_date="2014-01-01", end_date="2024-06-30", fields = "CLOSE", interval="daily")
        except:
            continue
    df = pd.concat(stock_dict, axis=1)
    if i==0:
        df_full = df
    else:
        df_full = pd.concat([df_full,df], axis = 1) 

2024-10-09 15:55:17,828 P[42945] [MainThread 8694124352] UDF Core request failed. Gateway Time-out
2024-10-09 15:55:17,831 P[42945] [MainThread 8694124352] HTTP request failed: EikonError-UDF Core request failed. Gateway Time-out
2024-10-09 16:01:14,414 P[42945] [MainThread 8694124352] UDF Core request failed. Gateway Time-out
2024-10-09 16:01:14,421 P[42945] [MainThread 8694124352] HTTP request failed: EikonError-UDF Core request failed. Gateway Time-out
2024-10-09 16:03:44,009 P[42945] [MainThread 8694124352] UDF Core request failed. Gateway Time-out
2024-10-09 16:03:44,019 P[42945] [MainThread 8694124352] HTTP request failed: EikonError-UDF Core request failed. Gateway Time-out
2024-10-09 16:05:24,818 P[42945] [MainThread 8694124352] UDF Core request failed. Gateway Time-out
2024-10-09 16:05:24,827 P[42945] [MainThread 8694124352] HTTP request failed: EikonError-UDF Core request failed. Gateway Time-out
2024-10-09 16:05:47,078 P[42945] [MainThread 8694124352] UDF Core request failed

In [ ]:
df_full.to_csv('stocks_1.csv') 

In [ ]:
#data_df.assign(rics=rics_list, inplace = TRUE)